# Data Retrieval -- TripAdvisor Attractions

In [2]:
import json
import pandas as pd
import undetected_chromedriver as uc
from bs4 import BeautifulSoup
import time
import random
import os
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

Based on the previous notebooks, I now have a dataframe with all the attractions relevant for me. Next, I need to scrape the page of each attractions (based on the link in the data I scraped) to retrieve locations.

In [3]:
# Read in attractions
with open('tripadvisor_attractions_filtered.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

df = pd.DataFrame(data)
print(f"Loaded {len(df)} attractions")

Loaded 1112 attractions


Now I can scrape the pages, again using the undetected chromedriver.

In [10]:
# Same setup that worked before
driver = uc.Chrome()

# Visit homepage first to establish session
driver.get("https://www.tripadvisor.com")
time.sleep(random.uniform(4, 7))

os.makedirs('attraction_pages', exist_ok=True)

try:
    for i, row in df.iterrows():
        filename = f"attraction_pages/attraction_{i}.html"
        
        if os.path.exists(filename):
            print(f"Skipping {i+1}/{len(df)}: {row['name']} (already scraped)")
            continue
        
        print(f"Scraping {i+1}/{len(df)}: {row['name']}")
        
        try:
            driver.get(row['link'])
            time.sleep(random.uniform(3, 6))
            
            with open(filename, 'w', encoding='utf-8') as f:
                f.write(driver.page_source)
            
            print(f"  → Saved to {filename}")
        
        except Exception as e:
            print(f"  → Error: {e}")
        
        if (i + 1) % 10 == 0:
            scraped = len([f for f in os.listdir('attraction_pages') if f.endswith('.html')])
            print(f"\n--- Checkpoint: {scraped}/{len(df)} pages scraped so far ---\n")

finally:
    driver.quit()
    scraped = len([f for f in os.listdir('attraction_pages') if f.endswith('.html')])
    print(f"\nDone! {scraped}/{len(df)} pages saved")

Skipping 1/1112: Hallgrimskirkja (already scraped)
Skipping 2/1112: Perlan (already scraped)
Skipping 3/1112: Glacier Lagoon (already scraped)
Skipping 4/1112: Gullfoss Falls (already scraped)
Skipping 5/1112: Sky Lagoon (already scraped)
Skipping 6/1112: Skogafoss (already scraped)
Skipping 7/1112: Blue Lagoon (already scraped)
Skipping 8/1112: Harpa Concert Hall and Conference Centre (already scraped)
Skipping 9/1112: Thingvellir National Park (already scraped)
Skipping 10/1112: Seljalandsfoss waterfall (already scraped)
Skipping 11/1112: Reynisfjara Beach (already scraped)
Skipping 12/1112: Víkurfjara Black Sand Beach (already scraped)
Skipping 13/1112: Godafoss (already scraped)
Skipping 14/1112: Strokkur (already scraped)
Skipping 15/1112: Sun Voyager (already scraped)
Skipping 16/1112: Reykjadalur Hot Springs (already scraped)
Skipping 17/1112: Golden Circle Route (already scraped)
Skipping 18/1112: National Museum of Iceland (already scraped)
Skipping 19/1112: Dyrholaey (already

I have again saved the html files for all of them, which I will next use to extract the information I need.